<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Portrait Segmentation (Portrait128) — Akida 2 Benchmark

Run Time: ~35 minutes on the Akida 2 FPGA

This notebook walks through evaluating and benchmarking a model on the **Portrait128** dataset, targeting Akida 2 hardware.

For details on the dataset and preparation of the model, see the neighbouring README.md and `portrait128_notebook_training.ipynb`.

> **Note:** the hardware benchmark needs a physical Akida 2 FPGA device. Every hardware step is guarded, so the notebook runs end to end without a board: the latency cells are skipped, and sparsity is still computed on the software backend. It will not produce hardware numbers on Colab.

The Akida 2 reference platform is an FPGA running at 25 MHz. Alongside the measured latency, the notebook reports a projected latency at the AKD2500 target clock. Power is not measured yet, so the benchmark is latency-only.

## Setup

In [1]:
import os
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/portrait128'
INPUT_SHAPE = (128, 128, 3)

# Pretrained models committed to the repo. If you trained your own with the
# training notebook or portrait128_train.sh, set MODELS_DIR = './models/'.
MODELS_DIR = './pretrained_models/'

# The converted 8-bit (QAT) model produced by the training pipeline
MODEL_FILENAME = 'akida_unet_portrait128_i8_w8_a8_qat.fbz'

# Clocks (see portrait128_benchmark.py and brainchip_utils.hardware_utils.AKIDA_CLOCKS_HZ).
MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

## Model

The notebook loads a converted Akida model (`.fbz`) from `pretrained_models/`, for example
`pretrained_models/akida_unet_portrait128_i8_w8_a8_qat.fbz`. Pretrained models are committed
through `git-lfs`; see the [Trained models](../../../README.md#trained-models) section of the
top-level README for set-up.

The model takes a uint8 RGB image of 128 × 128 and returns one logit per pixel, as a
`(128, 128, 1)` map. Its first layer, `InputConv2D`, runs on Akida's dedicated input
processor, which takes the uint8 image directly.

In [2]:
ak_model = akida.Model(os.path.join(MODELS_DIR, MODEL_FILENAME))
ak_model.summary()

                  Model Summary                  
_________________________________________________
Input shape    Output shape   Sequences  Layers
[128, 128, 3]  [128, 128, 1]  1          36    
_________________________________________________

_____________________________________________________________________________
Layer (type)                               Output shape    Kernel shape    

====================== SW/conv_0-dequantizer (Software) =====================

conv_0 (InputConv2D)                       [64, 64, 16]    (3, 3, 3, 16)   
_____________________________________________________________________________
conv_1 (Conv2D)                            [64, 64, 32]    (3, 3, 16, 32)  
_____________________________________________________________________________
conv_2 (Conv2D)                            [32, 32, 64]    (3, 3, 32, 64)  
_____________________________________________________________________________
conv_3 (Conv2D)                            [32, 32, 64]  

## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the latency benchmark below is skipped.

In [3]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found — benchmark will be skipped.')
else:
    print('Akida device found:', device)

Target Akida device found
Akida device found: <akida.core.HardwareDevice object at 0x74d78ad6fc30>


## Samples

Akida's standard neural processors only do work for non-zero activations, so latency depends on the input. Random or synthetic data drives unrepresentative activity and gives misleading timings, so the benchmark runs on real images from the training split.

In [4]:
from portrait128_data import get_samples

NUM_SAMPLES = 100
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)

## Latency benchmark

The model is benchmarked in each of the three mapping modes, which trade the number of neural processors (NPs) used against the number of passes:

* **Minimal** maps the model onto the fewest NPs that can hold it.
* **AllNps** spreads it over as many NPs as possible, while keeping the minimum number of passes.
* **HwPr** (hardware partial reconfiguration) splits the model into several passes so each layer can use more NPs, reloading weights between passes on every inference.

Which mode is fastest depends on the model, so all three are measured.

The FPGA runs at 25 MHz, far below production silicon. The number of clock cycles per inference is fixed for a given model and mapping, whatever the clock, so the latency at the AKD2500 target clock is an exact rescale: `projected_ms = mean_inf_clk / PROJECTED_CLOCK * 1000`.

In [5]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    for mm in ['Minimal', 'AllNps', 'HwPr']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        projected_ms = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(ak_model)
        print(f'[{mm}] NPs={num_nps} passes={num_passes} '
              f'latency@25MHz={res["mean_clk_ms"]:.3f} ms '
              f'projected@1000MHz={projected_ms:.3f} ms')
else:
    print('Hardware not available — skipping latency benchmark.')

Power measurement not available: only supported on AKD1500 (IpVersion.v1).
  Repeat 1/10 done.
  Repeat 2/10 done.
  Repeat 3/10 done.
  Repeat 4/10 done.
  Repeat 5/10 done.
  Repeat 6/10 done.
  Repeat 7/10 done.
  Repeat 8/10 done.
  Repeat 9/10 done.
  Repeat 10/10 done.

  Mean inference time:    405.299 ms  (σ=7.030 ms)
  Mean on-chip time:      396.036 ms  (9900888 clocks)
  Total inferences run:   1000
[Minimal] NPs=67 passes=3 latency@25MHz=396.036 ms projected@1000MHz=9.901 ms
Power measurement not available: only supported on AKD1500 (IpVersion.v1).
  Repeat 1/10 done.
  Repeat 2/10 done.
  Repeat 3/10 done.
  Repeat 4/10 done.
  Repeat 5/10 done.
  Repeat 6/10 done.
  Repeat 7/10 done.
  Repeat 8/10 done.
  Repeat 9/10 done.
  Repeat 10/10 done.

  Mean inference time:    404.907 ms  (σ=7.040 ms)
  Mean on-chip time:      395.646 ms  (9891155 clocks)
  Total inferences run:   1000
[AllNps] NPs=69 passes=3 latency@25MHz=395.646 ms projected@1000MHz=9.891 ms
Power measurement

## Per-layer benchmark & sparsity

The per-layer benchmark (Minimal mapping) shows where the time goes, and the sparsity printout shows why. A layer's cost grows with the number of non-zero activations it receives, so on Akida, per-layer latency and input sparsity are linked: a layer with dense input and a large feature map is slow.

In this model, the slowest layers are the encoder's first full convolutions (`conv_1` to `conv_3`), which apply 3 × 3 kernels across every channel of the 64 × 64 and 32 × 32 feature maps. The decoder's upsampling layers, at up to 128 × 128, come next, and the deep encoder layers at 8 × 8 and 4 × 4 cost little.

The input layer (`conv_0`) is the exception to the sparsity rule: it runs on the dedicated input processor, whose cost doesn't depend on how many of its inputs are zero.

Sparsity doesn't need hardware, so it is also computed on the software backend when no device is present.

In [6]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
    print('Per-layer benchmark complete.')
else:
    # Sparsity can still be computed on the software backend without hardware.
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    print('Hardware not available — skipped latency; sparsity computed on software backend.')


Layer               Sparsity
----------------------------
conv_0               26.02%
conv_1               41.78%
conv_2               49.98%
conv_3               35.52%
dw_separable_4       11.18%
pw_separable_4       27.09%
dw_separable_5        9.95%
pw_separable_5       59.14%
dw_separable_6       13.68%
pw_separable_6       42.10%
dw_separable_7       13.74%
pw_separable_7       49.42%
dw_separable_8       19.40%
pw_separable_8       54.57%
dw_separable_9       20.78%
pw_separable_9       62.35%
dw_separable_10      25.13%
pw_separable_10      69.91%
dw_separable_11      32.66%
pw_separable_11      48.60%
dw_separable_12      24.16%
pw_separable_12      78.75%
dw_separable_13      39.64%
pw_separable_13      70.33%
dw_sepconv_t_0       66.37%
pw_sepconv_t_0       55.17%
dw_sepconv_t_1       54.85%
pw_sepconv_t_1       53.14%
dw_sepconv_t_2       46.27%
pw_sepconv_t_2       49.41%
dw_sepconv_t_3       41.03%
pw_sepconv_t_3       49.47%
dw_sepconv_t_4       41.18%
pw_sepconv_t_4   